# Prove the cumulative handoff

Use the selected alpha, validated stage and active batch path together. The reference alpha=0.01 is selected from the fixed validation matrix; it is not a universal hyperparameter recommendation.

Run all cells top to bottom. **Colab:** connect a fresh CPU / Python 3.12 runtime; the first cell clones the pinned GitHub source and installs its locked runtime dependencies. **Local:** run `uv sync --locked` from `module-a` and select that `.venv` interpreter. See [Colab and environment guidance](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/docs/colab.md). Every run creates a new temporary tracking directory and prints its path; download evidence before ending a hosted runtime. No API key is needed. These are complete worked references; independent practices have separate starting states. IDs and timing vary.

In [1]:
from pathlib import Path
import json, tempfile, sys, subprocess
PIN = {'package': 'campus-mlops', 'version': '0.1.1', 'code_revision': 'sha256:a7a3061850f1f1eaa158754e719262b26a000af73019ada402be743c6f66e6c2', 'repository': 'https://github.com/richhiey/ml-llm-ops.git', 'source_commit': 'eec754e0b1379fe8bb01e23543b92e036b4c6b62', 'package_path': 'module-a'}
assert sys.version_info[:2] == (3, 12), 'Use Python 3.12; see the linked setup guide.'
if 'google.colab' in sys.modules:
    # Start before importing numerical packages: installed files cannot replace live modules.
    expected = {'numpy':'2.3.3', 'pandas':'2.3.3', 'sklearn':'1.7.2', 'mlflow':'3.5.1'}
    for name, version in expected.items():
        if name in sys.modules and getattr(sys.modules[name], '__version__', None) != version:
            raise RuntimeError(f'{name} is already loaded at another version. Delete/reconnect the runtime, then run setup first.')
    checkout = Path('/content') / ('ml-llm-ops-' + PIN['source_commit'][:12])
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--no-checkout', PIN['repository'], str(checkout)], check=True)
    subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', PIN['source_commit']], check=True)
    actual = subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'], text=True).strip()
    assert actual == PIN['source_commit']
    package = checkout / PIN['package_path']
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(package/'requirements-colab.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(package)], check=True)
    # A running kernel does not reload newly installed editable .pth files.
    sys.path.insert(0, str(package / 'src'))
import campus_mlops
from campus_mlops import *
from campus_mlops.lifecycle import code_revision, MODEL_NAME
assert campus_mlops.__version__ == PIN['version'] == '0.1.1'
assert code_revision() == PIN['code_revision'], 'Helper changed: restore the pinned reference source.'
state = Path(tempfile.mkdtemp(prefix='campus-module-a-'))
print('Helper', campus_mlops.__version__, 'source commit:', PIN['source_commit'], 'state:', state)


Helper 0.1.1 source commit: eec754e0b1379fe8bb01e23543b92e036b4c6b62 state: /var/folders/tc/l_pmm63j3jn1y1vyr31_4qhc0000gn/T/campus-module-a-7bjiikkn


## Integrate an end-to-end run
Build from a new state with guard, explicit metrics, autologging and registry evidence. No orchestration framework is needed for this sequence.

In [2]:
ws=Workspace(state/'handoff')
record=ws.run(Config(alpha=0.01,range_guard=True,autolog=True),name='handoff')
version=register(ws,record['run_id'])
approve(ws,version,'Controlled alpha comparison and range guard checks')
print(activate(ws,version,'Final validated selection'))
output,inference=batch_predict(ws)
print(record); print(inference)
assert inference['rows']==24 and inference['complete_within_60s']

2026/10/19 14:42:21 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/19 14:42:21 INFO mlflow.store.db.utils: Updating database tables


2026-10-19 14:42:21 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-10-19 14:42:21 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


2026-10-19 14:42:21 INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step


...

2026/10/19 14:42:22 INFO mlflow.store.db.utils: Updating database tables


2026-10-19 14:42:22 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-10-19 14:42:22 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


Successfully registered model 'dev.linden-demand'.
Created version '1' of model 'dev.linden-demand'.


{'timestamp_utc': '2026-10-19T12:42:22.681318+00:00', 'model': 'dev.linden-demand', 'alias': 'active', 'previous_version': None, 'new_version': '1', 'run_id': 'c783379ae7f5420dbbaea276c98343a1', 'reason': 'Final validated selection'}
{'run_id': 'c783379ae7f5420dbbaea276c98343a1', 'config': {'alpha': 0.01, 'range_guard': True, 'autolog': True}, 'metrics': {'rmse_kwh': 1.4342817684933908, 'mae_kwh': 1.201891273831005}, 'evaluation_partition': 'validation', 'code_revision': 'sha256:a7a3061850f1f1eaa158754e719262b26a000af73019ada402be743c6f66e6c2', 'data_sha256': '97fd0648a8277e2ae88f90bb20389745667df3df6b363beef23787ff427e5e54', 'environment': {'python': '3.12.13', 'system': 'Darwin', 'machine': 'arm64', 'packages': {'alembic': '1.20.0', 'annotated-types': '0.8.0', 'anyio': '4.15.1', 'appnope': '1.0.0', 'asttokens': '3.0.2', 'attrs': '26.1.0', 'blinker': '1.9.0', 'cachetools': '6.2.6', 'cairocffi': '1.7.1', 'cairosvg': '2.8.2', 'campus-mlops': '0.1.1', 'certifi': '2026.10.18', 'cffi': '2.

## Reproduce before handoff
The recorded source/data/environment must match. Replaying to a new store tests absence of hidden registry and notebook dependencies.

In [3]:
report=reproduce(ws,record['run_id'],state/'fresh-reproduction')
print(report)
assert report['passed'] and report['lineage_matched']

2026/10/19 14:42:22 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/19 14:42:22 INFO mlflow.store.db.utils: Updating database tables


2026-10-19 14:42:22 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-10-19 14:42:22 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


2026-10-19 14:42:22 INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step


...

2026-10-19 14:42:22 INFO  [alembic.runtime.migration] Running upgrade 3da73c924c2f -> bf29a5ff90ea, add jobs table


2026-10-19 14:42:22 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-10-19 14:42:22 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


2026/10/19 14:42:22 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


{'original_run_id': 'c783379ae7f5420dbbaea276c98343a1', 'reproduced_run_id': '9f675144b6ed4e7fa26e198c88174e68', 'rmse_delta': 0.0, 'tolerance': 1e-06, 'passed': True, 'lineage_matched': True}


## Evaluate the reserved final holdout once
After selection, measure final rows without tuning against their result. Keep this run out of the validation comparison group.

In [4]:
final=ws.run(Config(alpha=0.01,range_guard=True,autolog=True),name='final-holdout',partition='final')
print(final['metrics'])
assert final['evaluation_partition']=='final'

2026/10/19 14:42:23 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


{'rmse_kwh': 1.2148472211584365, 'mae_kwh': 0.9971419477925263}


## Write a verifiable handoff index
The complete submission also includes your explained extension diff and setup instructions. Check each named file exists; an index is a locator, not proof by itself.

In [5]:
handoff={'version':version,'run_id':record['run_id'],'code_revision':record['code_revision'],
'data_sha256':record['data_sha256'],'config':record['config'],'environment':record['environment'],
'model_uri':record['model_uri'],'inference':inference,'reproduction':report,
'exact_training_command':record['command'],'tolerance_rmse_kwh':1e-6}
(state/'handoff.json').write_text(json.dumps(handoff,indent=2))
print(json.dumps(handoff,indent=2))
assert (ws.path/'batch.csv').exists() and (ws.path/'audit.jsonl').exists()

{
  "version": "1",
  "run_id": "c783379ae7f5420dbbaea276c98343a1",
  "code_revision": "sha256:a7a3061850f1f1eaa158754e719262b26a000af73019ada402be743c6f66e6c2",
  "data_sha256": "97fd0648a8277e2ae88f90bb20389745667df3df6b363beef23787ff427e5e54",
  "config": {
    "alpha": 0.01,
    "range_guard": true,
    "autolog": true
  },
  "environment": {
    "python": "3.12.13",
    "system": "Darwin",
    "machine": "arm64",
    "packages": {
      "alembic": "1.20.0",
      "annotated-types": "0.8.0",
      "anyio": "4.15.1",
      "appnope": "1.0.0",
      "asttokens": "3.0.2",
      "attrs": "26.1.0",
      "blinker": "1.9.0",
      "cachetools": "6.2.6",
      "cairocffi": "1.7.1",
      "cairosvg": "2.8.2",
      "campus-mlops": "0.1.1",
      "certifi": "2026.10.18",
      "cffi": "2.1.1",
      "charset-normalizer": "3.5.2",
      "click": "8.5.0",
      "cloudpickle": "3.1.2",
      "comm": "0.2.3",
      "contourpy": "1.4.0",
      "cryptography": "46.0.7",
      "cssselect2": "0.10.